In [1]:
import pandas as pd
import numpy as np
import networkx as nx
from pathlib import Path
from etc.parse_ids import XMLParser

In [2]:
# Get the notebook's directory and go up to project root
notebook_dir = Path().resolve()
project_root = notebook_dir.parent
data_folder = project_root / "data" / "resources"
PilotDC9 = data_folder / "PilotStudy_Afekta"

In [3]:
graph_gml = data_folder / "generated" / "modified_graph.gml"
human1_xml = data_folder / "Human-GEM.xml"

In [4]:
G = nx.read_gml(graph_gml)

In [5]:
chebi_match = pd.read_csv(PilotDC9 / "AFEKTA_chebis.csv")

In [6]:
name_match = pd.read_excel(PilotDC9 / "Supplementary_Material_2_Results_Dennisse_Avella.xlsx",sheet_name="keyIDs")

In [7]:
mask = name_match["Idlevel"] <= 2

# Clean empty values or '-' to None
curated = (
    name_match.loc[mask, "CuratedID"]
    .astype("string")
    .str.strip()
    .replace({"": pd.NA, "-": pd.NA})
)
name_match.loc[mask, "CuratedID"] = curated.where(curated.notna(), None)
# Clean chebi_match
chebi_clean = chebi_match.copy()
chebi_clean["Input name"] = (
    chebi_clean["Input name"]
    .astype("string")
    .str.strip()
    .replace({"": pd.NA, "-": pd.NA})
)
chebi_clean["CHEBI_ID"] = chebi_clean["CHEBI_ID"].replace({"": None, "-": None})
# Create lookup table from chebi_clean
lookup = (
    chebi_clean.dropna(subset=["Input name"])
    .drop_duplicates(subset=["Input name"], keep="first")
    .set_index("Input name")["CHEBI_ID"]
)
# Map Chebi_IDs to name_match
name_match.loc[mask, "Chebi_ID"] = name_match.loc[mask, "CuratedID"].map(lookup)

In [8]:
name_match

,Class,CuratedID,Idlevel,DatabaseID,Blood,Plasma,Mitra,Capitainer,Whatman,Chebi_ID
0,Amino acids and derivatives,1-methylhistidine,1.0,HMDB0000001,1,1,1,1,1,50599
1,Fatty acids and derivatives,2-Hydroxybutanoic acid,1.0,HMDB0000008 - HMDB0341410,1,1,1,1,1,1148
2,Other compounds,2-Piperidone,1.0,HMDB0011749,1,1,1,1,1,77761
3,Other compounds,3-Carboxy-4-methyl-5-propyl-2-furanpropionic acid,2.0,HMDB0061112,1,1,1,1,1,41254
4,Amino acids and derivatives,3-methylhistidine,2.0,HMDB0000479,1,1,1,1,1,27596
...,...,...,...,...,...,...,...,...,...,...
188,NaN,NaN,NaN,Amino acids and derivatives,1,1,1,1,1,NaN
189,NaN,NaN,NaN,Amino acids and derivatives,1,1,1,1,1,NaN
190,NaN,NaN,NaN,Amino acids and derivatives,1,1,1,1,1,NaN
191,NaN,NaN,NaN,Amino acids and derivatives,1,1,1,1,1,NaN


In [9]:
parser = XMLParser(human1_xml)

In [10]:
df = parser.to_identifier_df()

In [11]:
df

,chebi,smiles,inchikey,kegg,metanetx,vmhmetabolite,hmdb,lipidmaps,pubchem
HUMAN1_ID,,,,,,,,,
MAM00001c,carveol,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MAM00001e,carveol,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MAM00002c,appnn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MAM00002e,appnn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MAM00003c,CHEBI:78990,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
MAM01316n,CHEBI:60721,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MAM00767n,CHEBI:84503,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MAM01435m,C02528,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
